# Task 2 – Einen dynamischen Katalog vollständig erfassen
[Aufgabenübersicht](../README.md) · [Aufgabenbeschreibung](README.md) · [Demo 4](../../demos/04_dynamische_seiten_selenium.ipynb)

Du erstellst eine Beschaffungsliste: **alle verfügbaren Produkte mit einem Preis bis einschliesslich 30 CHF**. Eine Liste mit nur den anfangs sichtbaren Produkten wäre unvollständig. Ergänze Kategorie und Verfügbarkeit, kontrolliere den gesamten Katalog und dokumentiere die Herkunft.

Die Daten sind synthetisch. Voraussetzung ist der laufende Selenium-Dienst. Ersetze die vier mit TODO markierten Stellen. `NotImplementedError` ist eine absichtliche Bearbeitungsmarkierung. Ein unverändertes «Run All» stoppt an der ersten unvollständigen Stelle.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/webscraping_workshop/checks.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Öffne das Notebook innerhalb des Workshop-Repos.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import json
from datetime import datetime, timezone
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from webscraping_workshop.checks import fixture_server
from webscraping_workshop.runtime import browser_session, html_data_url

## A. Ausgangs-HTML untersuchen
Der folgende Abruf ist vorbereitet. Notiere, warum der leere Tabellenkörper keine leere Produktliste beweist.

In [ ]:
# Nur Python ruft diesen kurzlebigen HTTP-Server auf.
with fixture_server(ROOT / "data/teaching") as base_url:
    with requests.Session() as session:
        session.trust_env = False  # Loopback nicht über einen Firmenproxy schicken.
        response = session.get(f"{base_url}/dynamic_products.html", timeout=(3, 10))
        response.raise_for_status()
        response.encoding = "utf-8"
        raw_html = response.text

source_soup = BeautifulSoup(raw_html, "lxml")
raw_rows = source_soup.select("#products tbody tr")
print("Tabellenzeilen in der HTTP-Antwort:", len(raw_rows))
assert len(raw_rows) == 0, "Die Lehrseite sollte anfangs einen leeren Tabellenkörper haben."

## B. Fachliche Felder extrahieren
Implementiere `parse_products(html)`. Erwartet sind `id`, `name`, `category`, `price_chf` und `available`. Verwende für die Verfügbarkeit das Attribut `data-available`. Leere/fehlerhafte Daten sollen nicht als erfolgreicher Export erscheinen.

In [ ]:
def parse_products(html):
    # TODO: .name, .category, .price und data-id/data-available auslesen.
    # Erzeuge id, name, category, price_chf (Zahl), available (bool).
    # Fehlende Felder, ungültige Verfügbarkeit und doppelte IDs sollen einen Fehler auslösen.
    raise NotImplementedError("Implementiere parse_products.")

## C. Vollständigen Ladezustand erreichen
Implementiere `collect_complete_html(html, max_clicks=3)`. Warte zunächst auf mindestens eine Zeile und lies `data-total`. Lade weitere Produkte, solange die Gesamtzahl nicht erreicht ist. Nach jedem Klick muss die Zeilenzahl zunehmen. Nutze ein Klicklimit und beende die Browser-Sitzung auch bei Fehlern.

In [ ]:
def collect_complete_html(html, max_clicks=3):
    # TODO: Nutze with browser_session(ROOT) as browser: für verlässliches Aufräumen.
    # Öffne html_data_url(html), warte auf erste Zeilen, lies data-total aus.
    # Lade bis zur Gesamtzahl nach: klickbaren Button abwarten, klicken,
    # auf Zunahme der Zeilenzahl warten. Berücksichtige max_clicks.
    # Liefere (gerendertes_html, erwartete_gesamtzahl, anzahl_klicks) zurück.
    raise NotImplementedError("Implementiere collect_complete_html.")

In [ ]:
rendered_html, expected_total, clicks = collect_complete_html(raw_html)
products = parse_products(rendered_html)
assert len(products) == expected_total == 6
assert products["id"].is_unique
assert products["price_chf"].gt(0).all()
assert products["available"].dtype == bool
assert products["available"].sum() == 4
products

## D. Beschaffungsliste ableiten
Filtere auf verfügbare Produkte mit `price_chf <= 30`. Sortiere nach Preis und ID. Fasse anschliessend die Anzahl und Preissumme pro Kategorie zusammen. Die folgenden Prüfungen gelten ausdrücklich für die mitgelieferten Lehrdaten.

In [ ]:
# TODO: Wähle verfügbare Produkte bis einschliesslich 30 CHF.
# Sortiere nach Preis, dann ID, und setze den Index zurück.
# Fasse Anzahl und Preissumme pro Kategorie zusammen (category_summary).
raise NotImplementedError("Erzeuge selected und category_summary.")

In [ ]:
assert set(selected["id"]) == {"P101", "P103", "P105"}
assert len(selected) == 3
assert abs(selected["price_chf"].sum() - 63.90) < 1e-9
assert category_summary["product_count"].sum() == len(selected)
selected

## E. Export und Herkunft
Exportiere den vollständigen Katalog und die gefilterte Liste. Das Abrufdatum ist kein Preisgültigkeitsdatum; kennzeichne die Daten als synthetisch. Prüfe durch Zurücklesen, dass der Export stimmt.

In [ ]:
# TODO: Exportiere products und selected nach data/output/task02_dynamic_sample/.
# Dateien: all_products.csv, available_products.csv (UTF-8, ohne DataFrame-Index).
# Ergänze provenance.json mit Quelle, synthetischem Modus, UTC-Abrufzeit,
# Währung, Zeilenzahlen, Klickanzahl und Filter.
# Lies beide CSVs wieder ein und vergleiche sie mit den DataFrames.
raise NotImplementedError("Implementiere Export und Rückleseprüfung.")

## F. Methodenwahl begründen
Beantworte in eigenen Worten:
1. Weshalb ist `bool("false")` hier falsch?
2. Welche Prüfung verhindert einen fachlich unvollständigen Export?
3. Was bewirkt das Klicklimit und was soll beim Erreichen geschehen?
4. Wann würdest Du Requests/BeautifulSoup, wann Selenium einsetzen?
5. Die Lehrdaten stehen auch im Skript. Welche andere Extraktionsstrategie wäre möglich, und weshalb simulieren wir trotzdem den Browserablauf?
6. Woran erkennst Du bei einer fremden Website, dass ein leerer Treffer eher auf einen falschen Selektor als auf JavaScript zurückzuführen ist?

**Deine Antworten:** Ergänze hier eine Markdown-Zelle.

[↑ Aufgabenübersicht](../README.md) · [Demo 4](../../demos/04_dynamische_seiten_selenium.ipynb)